<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/rafael_cleaning/notebooks/colab/cnn_interpre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 CNN Interprétabilité - COVID-19 Classification

**Objectif:** Analyser l'interprétabilité d'un modèle CNN (InceptionV3) sur radiographies COVID-19

**Méthodes d'interprétabilité:**
- **Grad-CAM** - Zones d'attention du modèle
- **LIME** - Explication par super-pixels
- **SHAP** - Valeurs de Shapley au niveau pixel

**Dataset:** COVID-19 Radiography (4 classes)
- COVID
- Normal
- Lung_Opacity
- Viral Pneumonia

**Modèle:** InceptionV3 pré-entraîné (du notebook transfer_learning_colab_revamp)

## 1. Initialisation

### Configuration Standalone

**⚠️ Cellule de configuration autonome**

In [ ]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. Les variables sont prêtes à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- Tous les imports des transformers

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    # Extraction models

    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])



    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Ajouter src/ au sys.path pour les imports
# src_path = str(project_root / 'src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
#     print(f"✅ Chemin src/ ajouté: {src_path}")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

# Exports pour compatibilité avec anciens notebooks
data_dir = config.data_dir
categories = config.classes
img_size = config.img_size


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB
# =============================================================================

plt.rcParams['figure.figsize'] = (15, 10)
sns.set_style('whitegrid')

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 70)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 70)
print(f"📂 Projet: {project_root}")
print(f"📊 Dataset: {data_dir}")
print(f"🏷️ Classes: {', '.join(categories)}")
print(f"🎛️ Images: {img_size}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if data_dir.exists() else '❌'}")
print("=" * 70)
print("\n💡 Variables disponibles:")
print("   • config: Configuration complète (Config object)")
print("   • ENV: Environnement actuel")
print("\n🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 70)


### Vérification GPU

In [ ]:
print("=" * 70)
print("VÉRIFICATION GPU")
print("=" * 70)

# Vérifier GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU disponible: {len(gpus)} GPU(s)")
    for gpu in gpus:
        print(f"   • {gpu.name}")

    # Configurer la mémoire GPU
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   • Memory growth activé")
else:
    print("\n⚠️ Pas de GPU disponible - utilisation CPU")
    print("   Pour activer GPU sur Colab: Runtime > Change runtime type > GPU")

print(f"\nTensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. Chargement des Données

In [ ]:
print("=" * 70)
print("CHARGEMENT DES DONNÉES")
print("=" * 70)

# Nombre d'images par classe (Colab Pro peut gérer plus)
N_IMAGES_PER_CLASS = 1345  # Ajuster selon GPU disponible, if None = toutes les images

image_paths = []
labels = []
labels_int = []

for idx, cat in enumerate(categories):
    cat_path = data_dir / cat / 'images'
    if cat_path.exists():
        imgs = sorted(list(cat_path.glob('*.png')))
        if N_IMAGES_PER_CLASS:
            imgs = imgs[:N_IMAGES_PER_CLASS]
        
        image_paths.extend(imgs)
        labels.extend([cat] * len(imgs))
        labels_int.extend([idx] * len(imgs))
        print(f"  {cat:20s}: {len(imgs):4d} images")

labels_int = np.array(labels_int)

print(f"\n  Total: {len(image_paths)} images")
print(f"  Classes: {len(categories)}")
print(f"  Distribution: {np.bincount(labels_int)}")

### Preprocessing Pipeline

In [ ]:
print("=" * 70)
print("PREPROCESSING PIPELINE")
print("=" * 70)

# Pipeline complet: Load + Resize + Mask
prep_pipeline = Pipeline([
    ('load', ImageLoader(color_mode='RGB', verbose=False)),
    ('resize', ImageResizer(img_size=(224, 224), verbose=False)),
    ('mask', ImageMasker(mask_paths=mask_paths, verbose=False))
])

print("\n⏳ Chargement et preprocessing des images...")
images = prep_pipeline.fit_transform(image_paths)
images = images.astype('float32')

print(f"\n📊 Images préparées:")
print(f"  Shape: {images.shape}")
print(f"  Range: [{images.min():.1f}, {images.max():.1f}]")
print(f"  Dtype: {images.dtype}")
print("\n⚠️ La normalisation InceptionV3 sera appliquée lors de l'analyse")

### Train/Test Split

In [ ]:
print("=" * 70)
print("TRAIN/TEST SPLIT")
print("=" * 70)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    images, labels_int,
    test_size=0.2,
    random_state=config.random_seed,
    stratify=labels_int
)

print(f"\nTrain: {X_train.shape[0]} images")
print(f"  Distribution: {np.bincount(y_train)}")
print(f"\nTest: {X_test.shape[0]} images")
print(f"  Distribution: {np.bincount(y_test)}")

### Visualisation des Données

In [ ]:
# Visualiser quelques échantillons
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    idx = np.random.randint(0, len(X_test))
    img = X_test[idx]
    label = y_test[idx]
    
    axes[i].imshow(img.astype('uint8'))
    axes[i].set_title(f'{categories[label]}', fontsize=11, weight='bold')
    axes[i].axis('off')

plt.suptitle('Échantillon du Test Set (après masking)', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

## 3. Entraînement d'un CNN Custom

## 3. Chargement du Modèle InceptionV3

In [ ]:
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

print("=" * 70)
print("CHARGEMENT DU MODÈLE INCEPTIONV3")
print("=" * 70)

model_path = config.models_dir / 'inceptionv3_best.keras'

if model_path.exists():
    print(f"\n📥 Chargement: {model_path.name}")
    model_inception = keras.models.load_model(model_path)
    print("✅ Modèle chargé")
    
    print(f"\n📐 Architecture:")
    print(f"  Input: {model_inception.input_shape}")
    print(f"  Output: {model_inception.output_shape}")
    print(f"  Paramètres: {model_inception.count_params():,}")
else:
    print(f"\n❌ Modèle non trouvé: {model_path}")
    print("   Veuillez d'abord exécuter: transfer_learning_colab_revamp.ipynb")

### Évaluation du Modèle

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("=" * 70)
print("ÉVALUATION SUR TEST SET")
print("=" * 70)

# Prétraitement InceptionV3
X_test_preprocessed = inception_preprocess(X_test.copy())

# Prédictions
print("\n⏳ Calcul des prédictions...")
y_pred_proba = model_inception.predict(X_test_preprocessed, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Métriques
acc = accuracy_score(y_test, y_pred)

print(f"\n📊 Résultats:")
print(f"  Accuracy: {acc:.4f}")
print(f"  Correct: {np.sum(y_pred == y_test)}/{len(y_test)}")

print(f"\n📋 Rapport par classe:")
print(classification_report(y_test, y_pred, target_names=categories, digits=4))

### Matrice de Confusion

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=categories, yticklabels=categories,
    ax=ax, cbar_kws={'label': 'Count'}
)

ax.set_title(f'Matrice de Confusion - InceptionV3\nAccuracy: {acc:.3f}', 
             fontsize=12, weight='bold')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)

plt.tight_layout()
plt.show()

## 4. Interprétabilité - Grad-CAM

### Import des modules

In [ ]:
print("=" * 70)
print("IMPORT DES MODULES D'INTERPRÉTABILITÉ")
print("=" * 70)

try:
    from src.interpretability.gradcam import GradCAM, visualize_gradcam, overlay_heatmap, visualize_gradcam_grid
    from src.interpretability.lime_explainer import LIMEImageExplainer
    from src.interpretability.shap_explainer import SHAPExplainer
    from src.interpretability.utils import plot_multiple_explanations
    print("\n✅ Modules d'interprétabilité importés")
except ImportError as e:
    print(f"\n⚠️ Erreur: {e}")

# Dossier pour les résultats
interp_dir = config.results_dir / 'interpretability'
interp_dir.mkdir(parents=True, exist_ok=True)
print(f"\n📂 Résultats: {interp_dir}")

### Initialisation Grad-CAM

In [ ]:
print("=" * 70)
print("GRAD-CAM - INCEPTIONV3")
print("=" * 70)

# Créer l'explainer
gradcam = GradCAM(model_inception)

# Couches disponibles
conv_layers = gradcam.get_available_layers()
print(f"\n📐 Couches conv disponibles: {len(conv_layers)}")
print(f"   Dernières couches:")
for layer in conv_layers[-5:]:
    print(f"   • {layer}")
print(f"\n✅ Couche utilisée: {gradcam.layer_name}")

### Exemples Grad-CAM

In [ ]:
# Sélectionner quelques images pour analyse
n_samples = 12
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)

print(f"🎯 Analyse de {n_samples} images:")
for idx in sample_indices[:6]:
    true_label = y_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    status = "✅" if true_label == pred_label else "❌"
    print(f"  {status} Image {idx}: {categories[true_label]:15s} → {categories[pred_label]:15s} ({confidence:.2%})")

In [ ]:
# Générer les heatmaps Grad-CAM
print("\n🔍 Génération des heatmaps Grad-CAM...")

heatmaps = []
sample_images = []
sample_class_names = []
sample_confidences = []

for idx in sample_indices:
    image = X_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    # Calculer la heatmap
    heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)
    
    heatmaps.append(heatmap)
    sample_images.append(image)
    sample_class_names.append(categories[pred_label])
    sample_confidences.append(confidence)

# Visualiser en grille
fig = visualize_gradcam_grid(
    np.array(sample_images),
    heatmaps,
    sample_class_names,
    confidences=sample_confidences,
    n_cols=4,
    figsize=(20, 15),
    save_path=interp_dir / 'gradcam_grid.png'
)
plt.show()

print("✅ Visualisations Grad-CAM générées")

## 5. Interprétabilité - LIME

### Initialisation LIME

In [ ]:
print("=" * 70)
print("LIME - INCEPTIONV3")
print("=" * 70)

# Créer l'explainer
lime_explainer = LIMEImageExplainer(
    model_inception.predict,
    segmentation_method='quickshift',
    num_samples=1000
)

print("\n✅ LIME Explainer créé")
print("   Méthode: quickshift")
print("   Échantillons: 1000")
print("\n⏳ Génération des explications (2-3 min)...")

### Visualisations LIME

In [ ]:
# Analyser 6 images avec LIME
lime_samples = 6
lime_indices = sample_indices[:lime_samples]

fig, axes = plt.subplots(lime_samples, 3, figsize=(15, lime_samples * 5))

for i, idx in enumerate(lime_indices):
    image = X_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    print(f"Image {i+1}/{lime_samples}: {categories[pred_label]} ({confidence:.2%})")
    
    # Explication LIME
    explanation = lime_explainer.explain_instance(
        image,
        top_labels=1,
        num_features=10,
        random_seed=42
    )
    
    # Obtenir image et masque
    temp, mask = explanation.get_image_and_mask(
        pred_label,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )
    
    ax_row = axes[i] if lime_samples > 1 else axes
    
    # Original
    ax_row[0].imshow(image.astype('uint8'))
    ax_row[0].set_title(f'Original\n{categories[pred_label]}', fontsize=10, weight='bold')
    ax_row[0].axis('off')
    
    # Masque
    ax_row[1].imshow(mask, cmap='Reds', alpha=0.8)
    ax_row[1].set_title('Régions Importantes', fontsize=10, weight='bold')
    ax_row[1].axis('off')
    
    # Explication
    ax_row[2].imshow(temp)
    ax_row[2].set_title(f'LIME\n{confidence:.2%}', fontsize=10, weight='bold')
    ax_row[2].axis('off')

plt.suptitle('LIME - InceptionV3', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(interp_dir / 'lime_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Explications LIME générées")

## 6. Interprétabilité - SHAP

### Initialisation SHAP

In [ ]:
print("=" * 70)
print("SHAP - INCEPTIONV3")
print("=" * 70)

# Préparer données de référence (background)
n_background = 50
background_indices = np.random.choice(len(X_train), n_background, replace=False)
background_data = X_train[background_indices]

print(f"\n📊 Background: {background_data.shape}")

# Créer l'explainer
shap_explainer = SHAPExplainer(model_inception, background_data)

print("✅ SHAP Explainer créé")
print("⏳ Calcul des valeurs SHAP (3-5 min)...")

### Visualisations SHAP

In [ ]:
# Analyser 6 images avec SHAP
shap_samples = 6
shap_indices = sample_indices[:shap_samples]
shap_images = X_test[shap_indices]

# Calculer valeurs SHAP
shap_values = shap_explainer.explain(shap_images, check_additivity=False)

print("✅ Valeurs SHAP calculées")

# Visualiser
fig, axes = plt.subplots(shap_samples, 3, figsize=(15, shap_samples * 5))

for i, idx in enumerate(shap_indices):
    image = X_test[idx]
    pred_label = y_pred[idx]
    confidence = y_pred_proba[idx, pred_label]
    
    # Heatmaps SHAP
    shap_mean = np.mean(np.abs(shap_values[i][pred_label]), axis=-1)
    shap_signed = np.mean(shap_values[i][pred_label], axis=-1)
    
    ax_row = axes[i] if shap_samples > 1 else axes
    
    # Original
    ax_row[0].imshow(image.astype('uint8'))
    ax_row[0].set_title(f'Original\n{categories[pred_label]}', fontsize=10, weight='bold')
    ax_row[0].axis('off')
    
    # SHAP Magnitude
    im1 = ax_row[1].imshow(shap_mean, cmap='Reds')
    ax_row[1].set_title('SHAP\n(Magnitude)', fontsize=10, weight='bold')
    ax_row[1].axis('off')
    plt.colorbar(im1, ax=ax_row[1], fraction=0.046, pad=0.04)
    
    # SHAP Signed
    im2 = ax_row[2].imshow(shap_signed, cmap='RdBu_r', 
                           vmin=-np.abs(shap_signed).max(), 
                           vmax=np.abs(shap_signed).max())
    ax_row[2].set_title(f'SHAP (Signed)\n{confidence:.2%}', fontsize=10, weight='bold')
    ax_row[2].axis('off')
    plt.colorbar(im2, ax=ax_row[2], fraction=0.046, pad=0.04)

plt.suptitle('SHAP - InceptionV3', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(interp_dir / 'shap_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualisations SHAP générées")

## 7. Comparaison des Méthodes

In [ ]:
print("=" * 70)
print("COMPARAISON DES 3 MÉTHODES")
print("=" * 70)

# Première image
comp_idx = sample_indices[0]
comp_image = X_test[comp_idx]
comp_pred = y_pred[comp_idx]
comp_conf = y_pred_proba[comp_idx, comp_pred]

print(f"\n🔍 Analyse:")
print(f"   Classe: {categories[comp_pred]}")
print(f"   Confiance: {comp_conf:.2%}")

# 1. Grad-CAM
gradcam_heatmap = gradcam.compute_heatmap(comp_image, class_idx=comp_pred)

# 2. LIME
lime_explanation = lime_explainer.explain_instance(
    comp_image, 
    top_labels=1, 
    num_features=5,
    random_seed=42
)

# 3. SHAP
shap_single = shap_explainer.explain(comp_image[np.newaxis, ...], check_additivity=False)
shap_values_single = shap_single[0][comp_pred]

# Visualiser
fig = plot_multiple_explanations(
    comp_image,
    gradcam_heatmap=gradcam_heatmap,
    lime_explanation=lime_explanation,
    shap_values=shap_values_single,
    class_idx=comp_pred,
    class_name=categories[comp_pred],
    confidence=comp_conf,
    save_path=interp_dir / 'comparison_all_methods.png'
)
plt.show()

print("\n✅ Comparaison générée")

## 8. Analyse des Erreurs

In [ ]:
print("=" * 70)
print("ANALYSE DES ERREURS")
print("=" * 70)

# Images mal classées
incorrect_indices = np.where(y_pred != y_test)[0]

if len(incorrect_indices) > 0:
    print(f"\n❌ {len(incorrect_indices)} erreurs sur {len(y_test)} ({len(incorrect_indices)/len(y_test):.2%})")
    
    # Analyser 3 premières erreurs
    n_errors = min(3, len(incorrect_indices))
    
    fig, axes = plt.subplots(n_errors, 3, figsize=(15, n_errors * 5))
    if n_errors == 1:
        axes = axes.reshape(1, -1)
    
    for i, idx in enumerate(incorrect_indices[:n_errors]):
        true_label = y_test[idx]
        pred_label = y_pred[idx]
        confidence = y_pred_proba[idx, pred_label]
        
        print(f"\nErreur {i+1}:")
        print(f"  Vrai: {categories[true_label]}")
        print(f"  Prédit: {categories[pred_label]} ({confidence:.2%})")
        
        image = X_test[idx]
        heatmap = gradcam.compute_heatmap(image, class_idx=pred_label)
        
        # Original
        axes[i, 0].imshow(image.astype('uint8'))
        axes[i, 0].set_title(f'Original\nVrai: {categories[true_label]}', fontsize=10, weight='bold')
        axes[i, 0].axis('off')
        
        # Heatmap
        axes[i, 1].imshow(heatmap, cmap='jet')
        axes[i, 1].set_title('Grad-CAM', fontsize=10, weight='bold')
        axes[i, 1].axis('off')
        
        # Overlay
        superimposed = overlay_heatmap(image, heatmap, alpha=0.4, colormap='jet')
        axes[i, 2].imshow(superimposed)
        axes[i, 2].set_title(f'Prédit: {categories[pred_label]}\n{confidence:.2%}', fontsize=10, weight='bold')
        axes[i, 2].axis('off')
    
    plt.suptitle('Analyse des Erreurs - Grad-CAM', fontsize=13, weight='bold')
    plt.tight_layout()
    plt.savefig(interp_dir / 'error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("\n✅ Aucune erreur!")

## 9. Conclusion

### 🔍 Méthodes utilisées

| Méthode | Vitesse | Type | Usage |
|---------|---------|------|-------|
| **Grad-CAM** | ⚡⚡⚡ Très rapide | Zones d'attention | Production, démo |
| **LIME** | ⚡⚡ Moyen | Super-pixels | Analyse exploratoire |
| **SHAP** | ⚡ Lent | Niveau pixel | Recherche, audit |

### 💡 Insights

**Le modèle se concentre sur:**
- ✅ Les poumons (zone principale)
- ✅ Les patterns d'opacité pour COVID
- ✅ Les structures pulmonaires pour Normal
- ⚠️ Parfois des artefacts (à surveiller)

### 📊 Résultats sauvegardés

Tous les fichiers sont dans: `results/interpretability/`
- `gradcam_grid.png` - Grille Grad-CAM
- `lime_explanations.png` - Explications LIME
- `shap_explanations.png` - Visualisations SHAP
- `comparison_all_methods.png` - Comparaison
- `error_analysis.png` - Analyse des erreurs

### 🎯 Recommandations

1. **Production**: Utiliser **Grad-CAM** (rapide, efficace)
2. **Audit médical**: **Grad-CAM + LIME**
3. **Recherche**: **SHAP** pour analyses rigoureuses

### 🔬 Validation

Les visualisations doivent être validées par des **radiologues** pour:
- Vérifier que le modèle regarde les bonnes zones anatomiques
- Identifier les biais potentiels
- Confirmer la pertinence clinique

---

**🎉 NOTEBOOK COMPLET !**

Vous avez:
1. ✅ Chargé et prétraité les données avec vos modules
2. ✅ Évalué le modèle InceptionV3
3. ✅ Analysé avec Grad-CAM, LIME et SHAP
4. ✅ Identifié les zones d'attention
5. ✅ Analysé les erreurs
6. ✅ Généré des rapports visuels

<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/rafael_cleaning/notebooks/colab/cnn_interpre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 CNN Interprétabilité - COVID-19 Classification

**Objectif :** Analyser l'interprétabilité d'un modèle CNN entraîné sur radiographies COVID-19

**Méthodes d'interprétabilité :**
- **Grad-CAM** (Gradient-weighted Class Activation Mapping) - Zones d'attention
- **LIME** (Local Interpretable Model-agnostic Explanations) - Super-pixels importants
- **SHAP** (SHapley Additive exPlanations) - Contributions au niveau pixel

**Dataset :** COVID-19 Radiography (4 classes)
- COVID
- Normal
- Lung_Opacity
- Viral Pneumonia

**Modèle analysé :** InceptionV3 (meilleur modèle du notebook transfer_learning_colab_revamp)

## 1. Initialisation

### Configuration Standalone

**⚠️ REMPLACER CETTE CELLULE** par le contenu complet de `CELL_CONFIG_STANDALONE.py`

### Vérification GPU